In [1]:
import requests
import pandas as pd
import os

# --- CONFIGURATION ---
START_DATE = "2021-01-01"
END_DATE = "2025-12-31"
WEATHER_FILENAME = "tn_weather_top10_21_25.csv"

# Verified Top 10 Cities (2012-2025)
CITIES = {
    "nashville": {"lat": 36.16, "lon": -86.78},
    "memphis": {"lat": 35.15, "lon": -90.05},
    "knoxville": {"lat": 35.96, "lon": -83.92},
    "chattanooga": {"lat": 35.04, "lon": -85.30},
    "clarksville": {"lat": 36.53, "lon": -87.36},
    "murfreesboro": {"lat": 35.85, "lon": -86.39},
    "franklin": {"lat": 35.93, "lon": -86.87},
    "johnson_city": {"lat": 36.31, "lon": -82.35},
    "jackson": {"lat": 35.61, "lon": -88.81},
    "hendersonville": {"lat": 36.30, "lon": -86.62}
}

def fetch_comprehensive_weather():
    print(f"Starting API call for {len(CITIES)} cities...")
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    hourly_params = [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "cloud_cover",
        "wind_speed_10m",
        "shortwave_radiation"
    ]
    
    params = {
        "latitude": [c["lat"] for c in CITIES.values()],
        "longitude": [c["lon"] for c in CITIES.values()],
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": ",".join(hourly_params),
        "temperature_unit": "fahrenheit",
        "wind_speed_unit": "mph",
        "precipitation_unit": "inch",
        "timezone": "America/Chicago"
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data_json = response.json()
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return None
    
    city_dfs = []
    for i, (name, _) in enumerate(CITIES.items()):
        city_data = data_json[i]['hourly']
        df = pd.DataFrame({'timestamp': pd.to_datetime(city_data['time'])})
        
        for param in hourly_params:
            df[f'{name}_{param}'] = city_data[param]
            
        city_dfs.append(df.set_index('timestamp'))
    
    # Concatenate all cities side-by-side
    full_df = pd.concat(city_dfs, axis=1).reset_index()
    return full_df

# --- EXECUTE AND SAVE ---
df_weather = fetch_comprehensive_weather()

if df_weather is not None:
    save_path = os.path.join(os.getcwd(), WEATHER_FILENAME)
    df_weather.to_csv(save_path, index=False)
    print(f"\nSUCCESS: Data saved to {save_path}")
    print(f"Rows: {len(df_weather)} | Columns: {len(df_weather.columns)}")

Starting API call for 10 cities...

SUCCESS: Data saved to /Users/garrett/Documents/CECS CAP 26/tn_weather_top10_21_25.csv
Rows: 43824 | Columns: 61


In [2]:
import pandas as pd

df = pd.read_csv("tn_weather_top10_21_25.csv")
df.head()

,timestamp,nashville_temperature_2m,nashville_relative_humidity_2m,nashville_precipitation,nashville_cloud_cover,nashville_wind_speed_10m,nashville_shortwave_radiation,memphis_temperature_2m,memphis_relative_humidity_2m,memphis_precipitation,...,jackson_precipitation,jackson_cloud_cover,jackson_wind_speed_10m,jackson_shortwave_radiation,hendersonville_temperature_2m,hendersonville_relative_humidity_2m,hendersonville_precipitation,hendersonville_cloud_cover,hendersonville_wind_speed_10m,hendersonville_shortwave_radiation
0,2021-01-01 00:00:00,44.0,94,0.004,100,8.3,0.0,47.0,95,0.087,...,0.020,100,12.0,0.0,44.1,92,0.008,100,8.0,0.0
1,2021-01-01 01:00:00,45.0,96,0.024,100,8.0,0.0,48.9,96,0.150,...,0.161,100,13.2,0.0,44.3,99,0.059,100,7.9,0.0
2,2021-01-01 02:00:00,46.0,97,0.031,100,9.4,0.0,51.0,96,0.146,...,0.283,100,14.1,0.0,45.3,99,0.028,100,9.9,0.0
3,2021-01-01 03:00:00,47.0,97,0.130,100,9.3,0.0,53.7,95,0.083,...,0.016,100,12.5,0.0,46.4,99,0.102,100,8.4,0.0
4,2021-01-01 04:00:00,49.1,97,0.130,100,11.1,0.0,56.8,96,0.142,...,0.012,100,12.1,0.0,48.2,100,0.138,100,10.2,0.0
